# 02 — Generate Task Default Params

Exports the `default_params()` of every registered pipeline task to `task_defaults.json`.

**When to run:** whenever a new task class is added to `pipeline_tasks/`.

**Workflow:**
1. Run all cells → `task_defaults.json` is written at the repo root.
2. Copy `task_defaults.json` → `pipeline_config.json` and edit your copy.
3. Use `pipeline_config.json` with `ConfigManager.load()` in the pipeline notebooks.

> The generated file is a **defaults snapshot** — do not edit it directly.

In [ ]:
import sys
from pathlib import Path

_repo_root = str(Path("..").resolve())
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import pipeline_tasks
from config_manager import ConfigManager

print("Imports OK")

## Discover tasks

Task classes are read from `pipeline_tasks.__all__`. Adding a new task to
`pipeline_tasks/__init__.py` is all that is needed — this notebook requires no edits.

In [ ]:
task_classes = [
    getattr(pipeline_tasks, name)
    for name in pipeline_tasks.__all__
]

print(f"Found {len(task_classes)} task(s):")
for cls in task_classes:
    print(f"  {cls.task_name!r}  deps={cls.dependencies}")

## Inspect default params

In [ ]:
for cls in task_classes:
    print(f"\n── {cls.task_name} ──")
    for k, v in cls.default_params().items():
        print(f"  {k}: {v!r}")

## Output path

Set `OUTPUT_JSON` to where the defaults snapshot should be written.
The file is always regenerated from scratch.

In [ ]:
OUTPUT_JSON = Path("../config/default_tasks_params.json")

## Generate JSON

In [ ]:
if OUTPUT_JSON.exists():
    OUTPUT_JSON.unlink()

cm = ConfigManager()
cm.set_global("data_root", "/path/to/raw/data")
cm.set_global("analysis_root", "/path/to/analysis")
for cls in task_classes:
    cm.register_task(cls)

cm.generate_template(OUTPUT_JSON)
print(f"Written: {OUTPUT_JSON.resolve()}")

## Verify output

In [ ]:
print(OUTPUT_JSON.read_text())